In [1]:
# A library of common molecules and their equilibrium geometries (in Angstroms)
molecule_library = {
    "hydrogen": "H 0.0 0.0 0.0; H 0.0 0.0 0.735",
    
    "ozone": (
        "O 0.0000  0.0000  0.0000; "
        "O 0.0000  1.0775  0.6865; "
        "O 0.0000 -1.0775  0.6865"
    ),
    
    "water": (
        "O 0.0000 0.0000 0.0000; "
        "H 0.0000 0.7572 -0.5865; "
        "H 0.0000 -0.7572 -0.5865"
    ),

    "carbon-dioxide": "C 0.0 0.0 0.0; O -1.16 0.0 0.0; O 1.16 0.0 0.0"
}

In [3]:
target_molecule = "carbon-dioxide"

selected_geometry = molecule_library[target_molecule]

from qiskit_nature.second_q.drivers import PySCFDriver
driver = PySCFDriver(atom=selected_geometry, basis="sto-3g")
raw_problem = driver.run()

from qiskit_nature.second_q.mappers import JordanWignerMapper
mapper = JordanWignerMapper()

from qiskit_nature.second_q.transformers import ActiveSpaceTransformer
transformer = ActiveSpaceTransformer(num_electrons=2, num_spatial_orbitals=2)
problem = transformer.transform(raw_problem)

from qiskit_nature.second_q.circuit.library import UCCSD, HartreeFock
ansatz = UCCSD(
    problem.num_spatial_orbitals,
    problem.num_particles,
    mapper,
    initial_state=HartreeFock(
        problem.num_spatial_orbitals,
        problem.num_particles,
        mapper,
    ),
)

import numpy as np
from qiskit_algorithms import VQE
from qiskit_algorithms.optimizers import SLSQP
from qiskit.primitives import StatevectorEstimator
vqe = VQE(StatevectorEstimator(), ansatz, SLSQP())
vqe.initial_point = np.zeros(ansatz.num_parameters)

from qiskit_algorithms import AdaptVQE
adapt_vqe = AdaptVQE(vqe)
adapt_vqe.supports_aux_operators = lambda: True  # temporary fix? <-

from qiskit_nature.second_q.algorithms import GroundStateEigensolver
solver = GroundStateEigensolver(mapper, adapt_vqe)

result = solver.solve(problem)

print(f"Total ground state energy = {result.total_energies[0]:.4f} Hartree")

Total ground state energy = -185.0668 Hartree
